In [1]:
import jieba
import jieba.posseg as pseg
import re

brand_dict = {"Adidas", "Nike", "华为", "小米", "苹果", "Zara", "ONLY", "优衣库", "三叶草", "李宁", "安踏"}
category_dict = {"连衣裙", "卫衣", "牛仔裤", "手机", "羽绒服", "T恤", "阔腿裤", "运动鞋", "跑鞋", "衬衫", "半身裙", "棉服", "风衣"}
attr_dict = {
    "color": {"白", "黑", "红", "蓝", "粉", "灰", "黄", "绿", "紫", "卡其", "米白", "白色", "黑色", "红色", "蓝色", "粉色", "灰色"},
    "gender": {"男鞋", "女鞋", "男款", "女款", "中性", "男", "女"},
    "year": {"2019新款", "2020新款", "2021新款", "2022新款", "2023新款", "新款"},
    "style": {"限量款", "联名款"}
}
for w in brand_dict | category_dict | set().union(*attr_dict.values()):
    jieba.add_word(w)

def extract_spec(text):
    specs = []
    # 容量：数字+GB/G/ml/L，避免匹配到单独的G（如5G网络）
    # 要求数字至少1位，后面跟单位，且单位前不能是数字（避免匹配256G中的G作为单位？实际上256G正确）
    # 改进：使用更严格的匹配，如 \d+(?:\.\d+)?\s*(?:GB|G|ml|L|毫升|升)(?!\w) 但G会匹配5G中的G，所以需要排除G前面是数字的情况？不，256G也是容量。问题在于“5G”中的5不是数字？5是数字。
    # 解决方案：如果匹配到的单位是单个G，且前面数字是1位并且不是常见容量组合（如8+128G），则可能误判。简单处理：先匹配多组合再匹配单G
    # 优先匹配 x+yG 格式
    combo = re.findall(r'\d+\s*[➕\+]\s*\d+\s*G', text)
    specs.extend(combo)
    # 再匹配普通容量，但排除掉已经匹配过的子串？简化：使用set去重
    normal = re.findall(r'\d+(?:\.\d+)?\s*(?:GB|ml|L|毫升|升)', text)
    specs.extend(normal)
    # 单独匹配 G，但要求前面数字至少2位或带小数点，避免匹配到5G中的5
    g_match = re.findall(r'\d{2,}(?:\.\d+)?\s*G\b', text)
    specs.extend(g_match)
    # 尺码
    size = re.findall(r'\b(?:S|M|L|XL|XXL|XXXL)\s*[码号]?', text)
    specs.extend(size)
    specs = list(set(specs))
    return specs

def ner_by_rule(text):
    words_pos = pseg.cut(text)
    entities = {"brand": [], "category": [], "spec": [], "attribute": []}
    entities["spec"].extend(extract_spec(text))
    for w, pos in words_pos:
        if w in brand_dict:
            entities["brand"].append(w)
        elif w in category_dict:
            entities["category"].append(w)
        elif w in attr_dict["color"]:
            entities["attribute"].append(f"颜色:{w}")
        elif w in attr_dict["gender"]:
            # 处理单字性别，如“男”“女”
            if w in ["男","女"]:
                entities["attribute"].append(f"性别:{w}款")
            else:
                entities["attribute"].append(f"性别:{w}")
        elif w in attr_dict["year"]:
            entities["attribute"].append(f"年份:{w}")
        elif w in attr_dict["style"]:
            entities["attribute"].append(f"款式:{w}")
        # 额外：如果w是单个字母且是尺码，也加入spec（因为extract_spec可能漏掉单独的S等）
        if w in ["S","M","L","XL"] and w not in entities["spec"]:
            entities["spec"].append(w)
    for k in entities:
        entities[k] = list(set(entities[k]))
    return entities

# 测试
test_titles = [
    "Adidas三叶草运动鞋男鞋2019新款",
    "华为P40手机8+128G黑色",
    "优衣库女装白色卫衣S码",
    "小米10 8+256G 5G手机"
]
for t in test_titles:
    print(t)
    print(ner_by_rule(t))
    print("-" * 50)

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\liuzi\AppData\Local\Temp\jieba.cache
Loading model cost 0.592 seconds.
Prefix dict has been built successfully.


Adidas三叶草运动鞋男鞋2019新款
{'brand': ['三叶草', 'Adidas'], 'category': ['运动鞋'], 'spec': [], 'attribute': ['年份:2019新款', '性别:男鞋']}
--------------------------------------------------
华为P40手机8+128G黑色
{'brand': ['华为'], 'category': ['手机'], 'spec': ['8+128G'], 'attribute': ['颜色:黑色']}
--------------------------------------------------
优衣库女装白色卫衣S码
{'brand': ['优衣库'], 'category': ['卫衣'], 'spec': ['S'], 'attribute': ['颜色:白色']}
--------------------------------------------------
小米10 8+256G 5G手机
{'brand': ['小米'], 'category': ['手机'], 'spec': ['256G', '8+256G'], 'attribute': []}
--------------------------------------------------


In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("cleaned_titles.csv")

# 构造伪类目标签（一级）
def get_level1(text):
    text = str(text).lower()
    if "连衣裙" in text:
        return "女装"
    elif "卫衣" in text:
        return "女装"
    elif "牛仔裤" in text or "阔腿裤" in text:
        return "女装"
    elif "半身裙" in text:
        return "女装"
    elif "手机" in text:
        return "数码"
    elif "笔记本" in text:
        return "数码"
    elif "羽绒服" in text or "棉服" in text:
        return "女装"
    else:
        return "其他"

df['level1'] = df['cleaned_text'].apply(get_level1)

# 去掉"其他"类以提高准确性（可选）
df = df[df['level1'] != '其他']
X = df['cleaned_text'].fillna('')
y = df['level1']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

vectorizer = TfidfVectorizer(max_features=3000, ngram_range=(1,2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_tfidf, y_train)
y_pred = clf.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

C:\Users\liuzi\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Accuracy: 0.7948717948717948
              precision    recall  f1-score   support

          女装       0.79      1.00      0.89       155
          数码       0.00      0.00      0.00        40

    accuracy                           0.79       195
   macro avg       0.40      0.50      0.44       195
weighted avg       0.63      0.79      0.70       195



In [3]:
import jieba
import re
from difflib import get_close_matches

# 常见错误映射
error_map = {
    "苹国": "苹果",
    "迖芙妮": "达芙妮",
    "卫衣衣": "卫衣",
    "连裙衣": "连衣裙",
    "运运鞋": "运动鞋",
    "休闭鞋": "休闲鞋"
}
correct_list = list(error_map.values()) + ["手机","连衣裙","卫衣","运动鞋","笔记本","羽绒服","牛仔裤"]

def correct_query(query):
    words = jieba.lcut(query)
    corrected = []
    for w in words:
        if w in error_map:
            corrected.append(error_map[w])
        else:
            candidates = get_close_matches(w, correct_list, n=1, cutoff=0.8)
            corrected.append(candidates[0] if candidates else w)
    return "".join(corrected)

def intent_classify(query):
    if re.search(r'(多少|什么|多大|几|颜色|容量|尺码|价格|多少钱|参数|配置)', query):
        return "属性查询"
    elif re.search(r'(推荐|哪款|什么牌子好|性价比高|最好|求推荐|哪个牌子|哪种)', query):
        return "推荐查询"
    else:
        return "商品查询"

def process_query(query):
    orig = query
    corr = correct_query(orig)
    seg = jieba.lcut(corr)
    intent = intent_classify(corr)
    return {"original": orig, "corrected": corr, "segmentation": seg, "intent": intent}

# 交互测试
while True:
    q = input("请输入Query（输入exit退出）：")
    if q.lower() == 'exit':
        break
    res = process_query(q)
    print(f"纠错后: {res['corrected']}")
    print(f"分词: {res['segmentation']}")
    print(f"意图: {res['intent']}")
    print("-"*40)

请输入Query（输入exit退出）：exit


In [4]:
import pandas as pd
import torch
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset

# ---------- 1. 读取数据 ----------
df = pd.read_csv("cleaned_titles.csv")

def get_pseudo_category(text):
    text = str(text).lower()
    if "连衣裙" in text or "卫衣" in text or "牛仔裤" in text or "阔腿裤" in text:
        return "女装"
    elif "手机" in text or "笔记本" in text:
        return "数码"
    else:
        return "其他"

df['label'] = df['cleaned_text'].apply(get_pseudo_category)
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['label'])
num_classes = len(le.classes_)
print("类别映射:", dict(zip(le.classes_, le.transform(le.classes_))))

# ---------- 2. 数据集类 ----------
class TitleDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer(text, truncation=True, padding='max_length', max_length=self.max_len, return_tensors='pt')
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# ---------- 3. 划分数据 ----------
X_train, X_test, y_train, y_test = train_test_split(df['cleaned_text'], df['label_encoded'], test_size=0.2, random_state=42)

# ---------- 4. 加载本地模型 ----------
model_path = "./bert-base-chinese-local"   # 改为你的本地路径
tokenizer = BertTokenizer.from_pretrained(model_path)
model = BertForSequenceClassification.from_pretrained(model_path, num_labels=num_classes)

train_dataset = TitleDataset(X_train.tolist(), y_train.tolist(), tokenizer)
test_dataset = TitleDataset(X_test.tolist(), y_test.tolist(), tokenizer)

# ---------- 5. 训练 ----------
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    evaluation_strategy="epoch",
    save_strategy="epoch"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()
eval_results = trainer.evaluate()
print("评估结果:", eval_results)

model.save_pretrained("./bert_category_model")
tokenizer.save_pretrained("./bert_category_model")

类别映射: {'其他': 0, '女装': 1, '数码': 2}


Some weights of the model checkpoint at ./bert-base-chinese-local were not used when initializing BertForSequenceClassification: ['cls.seq_relationship.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.bias']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were not initialized from the model check

Epoch,Training Loss,Validation Loss
1,0.124800,0.007659
2,0.011800,0.000166
3,0.002900,0.000073


***** Running Evaluation *****
  Num examples = 2000
  Batch size = 64
Saving model checkpoint to ./results\checkpoint-500
Configuration saved in ./results\checkpoint-500\config.json
Model weights saved in ./results\checkpoint-500\pytorch_model.bin
***** Running Evaluation *****
  Num examples = 2000
  Batch size = 64
Saving model checkpoint to ./results\checkpoint-1000
Configuration saved in ./results\checkpoint-1000\config.json
Model weights saved in ./results\checkpoint-1000\pytorch_model.bin
***** Running Evaluation *****
  Num examples = 2000
  Batch size = 64
Saving model checkpoint to ./results\checkpoint-1500
Configuration saved in ./results\checkpoint-1500\config.json
Model weights saved in ./results\checkpoint-1500\pytorch_model.bin


Training completed. Do not forget to share your model on huggingface.co/models =)


***** Running Evaluation *****
  Num examples = 2000
  Batch size = 64


Configuration saved in ./bert_category_model\config.json


评估结果: {'eval_loss': 7.306932093342766e-05, 'eval_runtime': 94.4997, 'eval_samples_per_second': 21.164, 'eval_steps_per_second': 0.339, 'epoch': 3.0}


Model weights saved in ./bert_category_model\pytorch_model.bin
tokenizer config file saved in ./bert_category_model\tokenizer_config.json
Special tokens file saved in ./bert_category_model\special_tokens_map.json


('./bert_category_model\\tokenizer_config.json',
 './bert_category_model\\special_tokens_map.json',
 './bert_category_model\\vocab.txt',
 './bert_category_model\\added_tokens.json')

In [5]:
from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

# 对测试集进行预测
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)
y_true = y_test.tolist()

# 计算指标
acc = accuracy_score(y_true, preds)
f1 = f1_score(y_true, preds, average='weighted')
print(f"测试集准确率: {acc:.4f}")
print(f"加权F1分数: {f1:.4f}")
print("\n详细分类报告:")
print(classification_report(y_true, preds, target_names=le.classes_))

***** Running Prediction *****
  Num examples = 2000
  Batch size = 64


测试集准确率: 1.0000
加权F1分数: 1.0000

详细分类报告:
              precision    recall  f1-score   support

          其他       1.00      1.00      1.00      1858
          女装       1.00      1.00      1.00       107
          数码       1.00      1.00      1.00        35

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



In [6]:
def predict_category(texts, model, tokenizer, le, max_len=128):
    """输入单个文本或文本列表，返回预测的类目名称"""
    if isinstance(texts, str):
        texts = [texts]
    encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_len, return_tensors='pt')
    with torch.no_grad():
        outputs = model(**encodings)
        preds = torch.argmax(outputs.logits, dim=1).numpy()
    return le.inverse_transform(preds)

# 示例测试
test_titles = ["Adidas三叶草运动鞋男鞋", "华为P40手机8+128G", "优衣库白色卫衣"]
results = predict_category(test_titles, model, tokenizer, le)
for title, cat in zip(test_titles, results):
    print(f"{title} -> {cat}")

Adidas三叶草运动鞋男鞋 -> 其他
华为P40手机8+128G -> 数码
优衣库白色卫衣 -> 女装


In [7]:
errors = [(X_test.iloc[i], y_true[i], preds[i]) for i in range(len(y_true)) if y_true[i] != preds[i]]
for text, true, pred in errors[:5]:
    print(f"文本: {text[:50]}... | 真实: {le.classes_[true]} | 预测: {le.classes_[pred]}")

In [9]:
import pandas as pd
import torch
import numpy as np
from transformers import BertTokenizer, BertForSequenceClassification, Trainer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset

# ------------------------------
# 1. 读取数据并预处理（与训练时完全一致）
# ------------------------------
df = pd.read_csv("cleaned_titles.csv")
# 填充缺失值，避免 NaN
df['cleaned_text'] = df['cleaned_text'].fillna('')

def get_pseudo_category(text):
    text = str(text).lower()
    if "连衣裙" in text or "卫衣" in text or "牛仔裤" in text or "阔腿裤" in text:
        return "女装"
    elif "手机" in text or "笔记本" in text:
        return "数码"
    else:
        return "其他"

df['label'] = df['cleaned_text'].apply(get_pseudo_category)
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['label'])
num_classes = len(le.classes_)

# 划分测试集（与训练时相同 random_state=42）
X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned_text'], df['label_encoded'], test_size=0.2, random_state=42
)
# 再次确保测试集无 NaN
X_test = X_test.fillna('')

# ------------------------------
# 2. 定义 BERT 数据集类（与训练时一致）
# ------------------------------
class TitleDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer(text, truncation=True, padding='max_length', max_length=self.max_len, return_tensors='pt')
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# ------------------------------
# 3. 加载已训练的 BERT 模型
# ------------------------------
model_path = "./bert_category_model"   # 你保存的模型目录
tokenizer = BertTokenizer.from_pretrained(model_path)
model = BertForSequenceClassification.from_pretrained(model_path)

test_dataset = TitleDataset(X_test.tolist(), y_test.tolist(), tokenizer)
trainer = Trainer(model=model)
predictions = trainer.predict(test_dataset)
bert_preds = np.argmax(predictions.predictions, axis=1)

bert_acc = accuracy_score(y_test, bert_preds)
bert_f1 = f1_score(y_test, bert_preds, average='weighted')
bert_report = classification_report(y_test, bert_preds, target_names=le.classes_)

# ------------------------------
# 4. 训练 TF-IDF + LR 模型
# ------------------------------
# 注意：训练集可能含 NaN，填充
X_train = X_train.fillna('')
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_tfidf, y_train)
lr_preds = lr.predict(X_test_tfidf)

lr_acc = accuracy_score(y_test, lr_preds)
lr_f1 = f1_score(y_test, lr_preds, average='weighted')
lr_report = classification_report(y_test, lr_preds, target_names=le.classes_)

# ------------------------------
# 5. 生成 evaluation_report.txt
# ------------------------------
with open("evaluation_report.txt", "w", encoding="utf-8") as f:
    f.write("=" * 60 + "\n")
    f.write("商品标题类目预测评估报告\n")
    f.write("=" * 60 + "\n\n")

    f.write("【数据集信息】\n")
    f.write(f"总样本数: {len(df)}\n")
    f.write(f"训练集大小: {len(X_train)}\n")
    f.write(f"测试集大小: {len(X_test)}\n")
    f.write(f"类别分布: {dict(df['label'].value_counts())}\n\n")

    f.write("【BERT模型 (fine-tune) 评估结果】\n")
    f.write(f"准确率 (Accuracy): {bert_acc:.4f}\n")
    f.write(f"加权F1分数 (Weighted F1): {bert_f1:.4f}\n")
    f.write("详细分类报告:\n")
    f.write(bert_report)
    f.write("\n")

    f.write("【TF-IDF + 逻辑回归 评估结果】\n")
    f.write(f"准确率 (Accuracy): {lr_acc:.4f}\n")
    f.write(f"加权F1分数 (Weighted F1): {lr_f1:.4f}\n")
    f.write("详细分类报告:\n")
    f.write(lr_report)
    f.write("\n")

    f.write("【Bad Case 分析 (BERT模型)】\n")
    errors = [(X_test.iloc[i], y_test.iloc[i], bert_preds[i]) for i in range(len(y_test)) if y_test.iloc[i] != bert_preds[i]]
    f.write(f"错误样本数: {len(errors)} / {len(y_test)} (错误率 {len(errors)/len(y_test):.2%})\n")
    f.write("前10个错误示例:\n")
    for text, true, pred in errors[:10]:
        f.write(f"文本: {text[:60]}... | 真实: {le.classes_[true]} | 预测: {le.classes_[pred]}\n")
    f.write("\n典型歧义案例: '苹果' 可能被误判 (若标题为'苹果水果'但模型学到'苹果手机'模式)\n")
    f.write("改进建议: 增加更细粒度的类别标注; 使用更大规模标注数据; 引入外部知识库消歧。\n")

print("✅ evaluation_report.txt 已生成")

# ------------------------------
# 6. 生成 model_comparison.csv
# ------------------------------
comparison_df = pd.DataFrame({
    "模型": ["TF-IDF + 逻辑回归", "BERT (fine-tune)"],
    "准确率": [lr_acc, bert_acc],
    "加权F1": [lr_f1, bert_f1],
    "训练时间(秒)": [2, 900],      # 可修改为实际记录的值
    "推理时间(ms/样本)": [0.2, 45]
})
comparison_df.to_csv("model_comparison.csv", index=False, encoding="utf-8-sig")
print("✅ model_comparison.csv 已生成")

loading file vocab.txt
loading file added_tokens.json
loading file special_tokens_map.json
loading file tokenizer_config.json
loading configuration file ./bert_category_model\config.json
Model config BertConfig {
  "_name_or_path": "./bert-base-chinese-local",
  "architectures": [
    "BertForSequenceClassification"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "directionality": "bidi",
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_2": 2
  },
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "pooler_fc_size": 768,
  "pooler_num_attention_heads": 12,
  "pooler_num_fc_layers": 3,
  "pooler_size_per_head": 128,
  "pooler

✅ evaluation_report.txt 已生成
✅ model_comparison.csv 已生成
